In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
train_data_path = "/work/home/maben/project/rec_sys/projects/Ali_CCP_REC/dataset/datasetsali_ccp_train.parquet"
eval_data_path = "/work/home/maben/project/rec_sys/projects/Ali_CCP_REC/dataset/datasetsali_ccp_val.parquet"
test_data_path = "/work/home/maben/project/rec_sys/projects/Ali_CCP_REC/dataset/datasetsali_ccp_test.parquet"

In [3]:
train_df = pd.read_parquet(train_data_path)

In [4]:
len(train_df)

42299905

In [5]:
len(train_df[train_df['click']==1])

1644256

In [4]:
train_df.head()

,click,purchase,101,121,122,124,125,126,127,128,...,127_14,150_14,D109_14,D110_14,D127_14,D150_14,D508,D509,D702,D853
0,0,0,1,1,1,1,1,0,1,1,...,1,1,0.4734,0.562,0.0856,0.1902,0.07556,0.000,0.0000,0.00000
1,0,0,1,1,1,1,1,0,1,1,...,1,1,0.4734,0.562,0.0856,0.1902,0.00000,0.000,0.0000,0.00000
2,1,0,1,1,1,1,1,0,1,1,...,1,1,0.4734,0.562,0.0856,0.1902,0.56050,0.256,0.4626,0.34400
3,0,0,1,1,1,1,1,0,1,1,...,1,1,0.4734,0.562,0.0856,0.1902,0.26150,0.000,0.0000,0.12213
4,0,0,1,1,1,1,1,0,1,1,...,1,1,0.4734,0.562,0.0856,0.1902,0.35910,0.000,0.0000,0.00000


In [7]:
train_df.columns

Index(['click', 'purchase', '101', '121', '122', '124', '125', '126', '127',
       '128', '129', '205', '206', '207', '210', '216', '508', '509', '702',
       '853', '301', '109_14', '110_14', '127_14', '150_14', 'D109_14',
       'D110_14', 'D127_14', 'D150_14', 'D508', 'D509', 'D702', 'D853'],
      dtype='object')

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from typing import Dict, List, Tuple, Optional
import pandas as pd
import numpy as np

In [6]:
class RankDataset(Dataset):

    def __init__(
        self,
        df,
        feature_cols: List[str],
        label_cols: List[str],
        user_col: str = 'user_id',
        item_col: str = 'item_id',
    ):
        self.feature_cols = feature_cols
        self.label_cols = label_cols
        self.user_col = user_col
        self.item_col = item_col

        # ---- 一次性转成 tensor，之后再也不碰 DataFrame ----
        self.user_ids = torch.from_numpy(
            df[user_col].values.astype(np.int64)
        ).long()

        self.item_ids = torch.from_numpy(
            df[item_col].values.astype(np.int64)
        ).long()

        # 按类型批量转换特征列
        self.feature_tensors = {}
        for col in feature_cols:
            values = df[col].values
            if values.dtype in (np.int32, np.int64, int):
                self.feature_tensors[col] = torch.from_numpy(
                    values.astype(np.int64)
                ).long()
            elif isinstance(values[0], (list, np.ndarray)):
                self.feature_tensors[col] = torch.from_numpy(
                    np.stack(values).astype(np.float32)
                ).float()
            else:
                self.feature_tensors[col] = torch.from_numpy(
                    values.astype(np.float32)
                ).float()

        # ---- 标签列转换 ----
        self.label_tensors = {}
        for col in label_cols:
            values = df[col].values
            # 标签通常是 0/1 二分类或连续值
            if values.dtype in (np.int32, np.int64, int, np.bool_):
                # 二分类标签：click、purchase 等
                self.label_tensors[col] = torch.from_numpy(
                    values.astype(np.float32)
                ).float()  # BCE loss 需要 float 类型
            else:
                # 连续标签：停留时长、评分等
                self.label_tensors[col] = torch.from_numpy(
                    values.astype(np.float32)
                ).float()

        self.size = len(df)

        # DataFrame 已经不需要了，释放内存
        del df

    def __len__(self):
        return self.size

    def __getitem__(self, idx) -> Dict[str, torch.Tensor]:
        # 纯索引操作，每次调用 < 1微秒
        sample = {
            'user_id': self.user_ids[idx],
            'item_id': self.item_ids[idx],
        }
        for col in self.feature_cols:
            sample[col] = self.feature_tensors[col][idx]

        labels = {}
        for col in self.label_cols:
            labels[col] = self.label_tensors[col][idx]

        return sample, labels

In [8]:
feature_cols = ["121", "122", "124", "125", "126", "127", "128", "129", "206", "207", "210", "216"]
label_cols = ['click','purchase']
user_col = "101"
item_col = "205"

In [9]:
dataset = RankDataset(train_df,feature_cols,label_cols,user_col,item_col)

In [10]:
dataset[0]

({'user_id': tensor(1),
  'item_id': tensor(0),
  '121': tensor(1),
  '122': tensor(1),
  '124': tensor(1),
  '125': tensor(1),
  '126': tensor(0),
  '127': tensor(1),
  '128': tensor(1),
  '129': tensor(1),
  '206': tensor(1),
  '207': tensor(1),
  '210': tensor(1),
  '216': tensor(1)},
 {'click': tensor(0.), 'purchase': tensor(0.)})